# Fish Audio S2 Pro — Two-Speaker Travel Dialogue (VM Jupyter, L4)

Self-hosts **open weights** (`fishaudio/s2-pro`) via the official fish-speech CLI on a **GCP GPU VM**. **Not** Colab. **Not** the paid fish.audio API. **Not** SGLang.

L4 (24GB, Ada) runs the flagship checkpoint in **bf16**. This is the quality A/B vs MOSS. T4 smoke: `fish_s2_t4_jupyter.ipynb`.

**Layout on the VM:** `~/fish-s2/{inputs,outputs,fish-speech}`

**From your Mac** (replace `ZONE`). First generate the Fish-curated script:

```bash
python run_fish_director.py chokhi_dhani.txt -o output/chokhi_dhani_fish.txt
```

Then:

```bash
VM=rotomax-gpu-20260819-081901
ZONE=us-east1-c   # or whatever gcloud compute instances list shows

gcloud compute ssh $VM --zone=$ZONE -- "mkdir -p ~/fish-s2/inputs ~/fish-s2/outputs"

gcloud compute scp \
  fish_s2_l4_jupyter.ipynb \
  output/chokhi_dhani_fish.txt \
  voices/refs/male_ref.wav \
  voices/refs/female_ref.wav \
  voices/refs/ref_texts.json \
  $VM:~/fish-s2/inputs/ --zone=$ZONE

gcloud compute ssh $VM --zone=$ZONE -- \
  "mv ~/fish-s2/inputs/fish_s2_l4_jupyter.ipynb ~/fish-s2/"

gcloud compute ssh $VM --zone=$ZONE -- -L 8080:localhost:8080
```

Open http://localhost:8080 → `fish-s2/fish_s2_l4_jupyter.ipynb`. **Kernel → Restart Kernel**, then run cells in order (1 → 8 smoke, then 9). Do not keep a messy in-place VM copy of this notebook.

After generate, on the Mac:

```bash
gcloud compute scp $VM:~/fish-s2/outputs/episode.wav . --zone=$ZONE
python run_mix_moss.py --episode episode.wav --cues output/chokhi_dhani_fish.cues.json
```

**SHARED CELLS:** install / parse / CLI helper must stay in lockstep with `fish_s2_t4_jupyter.ipynb`.

In [ ]:
# 1. GPU check (L4 quality — full bf16). Refuse T4.
import gc
import os
import subprocess
from pathlib import Path

import torch

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, (
    "No CUDA GPU. This notebook needs an L4 (or A100) VM with working nvidia-smi."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ROOT = Path.home() / "fish-s2"
INPUT_DIR = ROOT / "inputs"
OUT_DIR = ROOT / "outputs"
FISH_REPO = ROOT / "fish-speech"
for d in (ROOT, INPUT_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

gpu_name = torch.cuda.get_device_name(0)
capability = torch.cuda.get_device_capability(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

gc.collect()
torch.cuda.empty_cache()

try:
    print("nvidia-smi:", subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"],
        text=True,
    ).strip())
except Exception as e:
    print("nvidia-smi:", e)

print(f"GPU: {gpu_name} | VRAM {vram_gb:.1f} GB | capability={capability}")
print(f"ROOT={ROOT}")
assert vram_gb >= 22, (
    f"VRAM {vram_gb:.1f} GB < 22GB. This is the L4 notebook. "
    "Use fish_s2_t4_jupyter.ipynb on T4 (fp16 + chunks)."
)
assert capability >= (8, 0), (
    f"Compute capability {capability} < 8.0 (no native bf16). "
    "Use fish_s2_t4_jupyter.ipynb (--half)."
)
print("This notebook: bf16 (no --half), no outer 6-turn split, --no-compile on first run")

In [ ]:
# 2. Locate SCP'd inputs (no Colab upload)
# SHARED — patch both Jupyter notebooks if this cell breaks
import shutil
from pathlib import Path

ROOT = Path.home() / "fish-s2"
INPUT_DIR = ROOT / "inputs"
OUT_DIR = ROOT / "outputs"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# If files were scp'd to $HOME instead of ~/fish-s2/inputs, pull them in.
home = Path.home()
wanted = (
    "male_ref.wav",
    "female_ref.wav",
    "ref_texts.json",
)
for src in list(home.glob("*.txt")) + [home / n for n in wanted]:
    if src.is_file() and not (INPUT_DIR / src.name).exists():
        shutil.copy2(src, INPUT_DIR / src.name)
        print("copied from ~:", src.name)

scripts = sorted(p for p in INPUT_DIR.glob("*.txt") if p.name != "ref_texts.json")
assert scripts, (
    f"No directed .txt in {INPUT_DIR}. From Mac:\n"
    "  gcloud compute scp output/chokhi_dhani_fish.txt VM:~/fish-s2/inputs/ --zone=ZONE"
)
fish_scripts = sorted(INPUT_DIR.glob("*fish*.txt"))
SCRIPT_FILE = (fish_scripts or scripts)[0]
have_refs = (INPUT_DIR / "male_ref.wav").exists() and (INPUT_DIR / "female_ref.wav").exists()
assert have_refs, (
    f"Missing male_ref.wav / female_ref.wav in {INPUT_DIR}. "
    "SCP voices/refs/*.wav from the Mac."
)
print(f"Script: {SCRIPT_FILE.name}")
print("Refs: OK")
print(f"ref_texts.json: {(INPUT_DIR / 'ref_texts.json').exists()}")
print("inputs:", sorted(p.name for p in INPUT_DIR.iterdir()))

## Check refs
Listen before spending generate time. Refs set **timbre / accent / baseline energy**. Transcripts in `ref_texts.json` must match the wav words.

In [ ]:
# 3. Listen to refs
import IPython.display as ipd
for name in ("male_ref.wav", "female_ref.wav"):
    p = INPUT_DIR / name
    assert p.exists(), p
    print(name)
    display(ipd.Audio(str(p)))

In [ ]:
# 4. Install fish-speech deps (official CLI, not SGLang, not paid API)
# SHARED — if this cell errors, patch BOTH fish_s2_t4_jupyter.ipynb and fish_s2_l4_jupyter.ipynb
#
# Do NOT `pip install -e .` / `pip install .` on this DLVM:
# Ubuntu setuptools + vcs-versioning wants packaging>=26.2, OS has 21.3.
# Keep the VM's torch. Never install torchcodec (needs libnvrtc.so.13 / CUDA 13).
import re
import site
import subprocess
import sys
from pathlib import Path

import torch

ROOT = Path.home() / "fish-s2"
FISH_REPO = ROOT / "fish-speech"
ROOT.mkdir(parents=True, exist_ok=True)

if not (FISH_REPO / ".git").is_dir():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/fishaudio/fish-speech.git",
        str(FISH_REPO),
    ])
else:
    print("repo already cloned:", FISH_REPO)

user_site = Path(site.getusersitepackages())
user_site.mkdir(parents=True, exist_ok=True)
(user_site / "fish-speech.pth").write_text(str(FISH_REPO) + "\n")
print("PYTHONPATH via", user_site / "fish-speech.pth")

PIP = [sys.executable, "-m", "pip", "install", "--user"]
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "hydra-core", "hydra"])
subprocess.check_call(PIP + ["--no-cache-dir", "hydra-core==1.3.2", "omegaconf==2.3.0"])


def _pip_pkg(spec: str, *, only_binary: bool = True) -> None:
    print("installing", spec, flush=True)
    cmd = PIP + (["--only-binary=:all:", spec] if only_binary else [spec])
    if subprocess.call(cmd) != 0:
        if only_binary:
            print("  retry without only-binary:", spec, flush=True)
            subprocess.check_call(PIP + [spec])
        else:
            raise SystemExit(f"pip failed: {spec}")


for spec in (
    "click", "loguru", "einops", "natsort", "safetensors", "tqdm",
    "tiktoken", "pydantic>=2", "lightning", "librosa", "soundfile", "pydub",
    "loralib", "huggingface_hub", "einx", "transformers",
    "protobuf>=3.20,<6", "resampy", "ormsgpack", "cachetools",
):
    _pip_pkg(spec)

for spec in ("descript-audiotools", "descript-audio-codec"):
    _pip_pkg(spec)

# Match torchaudio to VM torch; do not pip-install torch.
ta_ver = torch.__version__.split("+")[0]
cuda = torch.version.cuda or "12.9"
cu_tag = "cu" + cuda.replace(".", "")
print(f"VM torch {torch.__version__} cuda {cuda} → torchaudio=={ta_ver} ({cu_tag})")
subprocess.check_call(
    PIP + ["--no-deps", f"torchaudio=={ta_ver}",
           "--index-url", f"https://download.pytorch.org/whl/{cu_tag}"]
)

# torchaudio 2.9 load() needs torchcodec (CUDA 13). Use soundfile instead.
infer_py = FISH_REPO / "fish_speech" / "models" / "text2semantic" / "inference.py"
src = infer_py.read_text(encoding="utf-8")
pat = re.compile(r"^(\s*)wav, sr = torchaudio\.load\(str\(audio_path\)\)\s*$", re.M)
if "sf.read(str(audio_path)" in src:
    print("encode_audio already uses soundfile")
else:
    m = pat.search(src)
    assert m, f"could not find torchaudio.load in {infer_py}"
    ind = m.group(1)
    repl = (
        f"{ind}import soundfile as sf\n"
        f"{ind}data, sr = sf.read(str(audio_path), dtype=\"float32\", always_2d=True)\n"
        f"{ind}wav = torch.from_numpy(data.T.copy())  # (channels, samples)"
    )
    infer_py.write_text(pat.sub(repl, src, count=1), encoding="utf-8")
    print("patched encode_audio → soundfile")

subprocess.check_call([
    sys.executable, "-c",
    "import hydra; assert hydra.__version__.startswith('1.3'), hydra.__version__; "
    "print('hydra', hydra.__version__)",
])
print("Installed. Repo:", FISH_REPO)

In [ ]:
# 5. Prefetch S2 Pro open weights (avoids mid-run downloads)
# SHARED — patch both notebooks if download/path breaks
from huggingface_hub import snapshot_download
from pathlib import Path

ROOT = Path.home() / "fish-s2"
FISH_REPO = ROOT / "fish-speech"
CKPT_DIR = FISH_REPO / "checkpoints" / "s2-pro"
MODEL_ID = "fishaudio/s2-pro"

CKPT_DIR.mkdir(parents=True, exist_ok=True)
if any(CKPT_DIR.rglob("codec.pth")):
    print("Weights already present — skip download")
else:
    print("Downloading", MODEL_ID, "→", CKPT_DIR)
    snapshot_download(MODEL_ID, local_dir=str(CKPT_DIR))
assert (CKPT_DIR / "codec.pth").exists() or any(CKPT_DIR.rglob("codec.pth")), (
    f"No codec.pth under {CKPT_DIR} — check the snapshot layout"
)
print("Weights cached. codec.pth:", next(CKPT_DIR.rglob("codec.pth"), CKPT_DIR / "codec.pth"))

In [ ]:
# 6. Config — L4 knobs (bf16, no outer chunk split). T4 notebook uses different values here only.
import json
from pathlib import Path

ROOT = Path.home() / "fish-s2"
INPUT_DIR = ROOT / "inputs"
OUT_DIR = ROOT / "outputs"
FISH_REPO = ROOT / "fish-speech"
CKPT_DIR = FISH_REPO / "checkpoints" / "s2-pro"
_codec = next(CKPT_DIR.rglob("codec.pth"), None)
assert _codec, f"No codec.pth under {CKPT_DIR} — re-run the weights cell"
CKPT_DIR = _codec.parent  # from_pretrained wants the dir that contains codec.pth
SPEAKER0_REF = INPUT_DIR / "male_ref.wav"
SPEAKER1_REF = INPUT_DIR / "female_ref.wav"
fish_scripts = sorted(INPUT_DIR.glob("*fish*.txt"))
SCRIPT_FILE = (
    fish_scripts
    or sorted(p for p in INPUT_DIR.glob("*.txt") if p.name != "ref_texts.json")
)[0]

ref_texts_path = INPUT_DIR / "ref_texts.json"
if ref_texts_path.exists():
    _texts = json.loads(ref_texts_path.read_text(encoding="utf-8"))
    REF_TEXT_M, REF_TEXT_F = _texts["male"], _texts["female"]
    print("Ref transcripts from ref_texts.json")
else:
    REF_TEXT_M = (
        "You know, this place has a real pull to it. I've been around a few spots "
        "like this, but here at Chokhee Dhaanee you can feel why people stay. "
        "Let me show you something interesting — take it at an easy pace with me."
    )
    REF_TEXT_F = (
        "Wait — that's actually fascinating. So this isn't only for show? "
        "I love how that changes how you look at Chokhee Dhaanee. "
        "Hah, okay, tell me about the Kalbeliya dancers next. I'm genuinely curious."
    )
    print("Using default ref transcripts — edit if your wav words differ.")

# L4-ONLY knobs
TURNS_PER_CHUNK = 0
CHUNK_LENGTH = 512
USE_HALF = False
USE_COMPILE = False  # set True after a clean smoke if you want kernel fusion
SMOKE_TURNS = 6
SEED = 42

assert SPEAKER0_REF.exists() and SPEAKER1_REF.exists()
assert CKPT_DIR.is_dir(), CKPT_DIR
print(f"Script: {SCRIPT_FILE.name}")
print(f"S0={SPEAKER0_REF.name} | S1={SPEAKER1_REF.name}")
print(
    f"TURNS_PER_CHUNK={TURNS_PER_CHUNK} CHUNK_LENGTH={CHUNK_LENGTH} "
    f"USE_HALF={USE_HALF} USE_COMPILE={USE_COMPILE}"
)

In [ ]:
# 7. Parse directed script → Fish <|speaker:N|> text
# SHARED — patch both notebooks if parse/token mapping breaks
import re

SPEAKER_LINE = re.compile(
    r"^\s*Speaker\s+(?P<num>1|2)\s*:\s*(?P<text>.+?)\s*$", re.IGNORECASE
)
TAG_LINE = re.compile(
    r"^\s*\[(?P<tag>M|F)\]\s*(?P<text>.+?)\s*$", re.IGNORECASE
)
# MOSS Speaker 1 / [M] → Fish speaker 0 (male_ref); Speaker 2 / [F] → 1
NUM_TO_FISH = {"1": "0", "2": "1"}
TAG_TO_FISH = {"M": "0", "F": "1"}


def parse_turns(text: str):
    turns = []
    for line in text.splitlines():
        if not line.strip():
            continue
        if m := SPEAKER_LINE.match(line):
            turns.append((NUM_TO_FISH[m.group("num")], m.group("text").strip()))
        elif m := TAG_LINE.match(line):
            turns.append((TAG_TO_FISH[m.group("tag").upper()], m.group("text").strip()))
    return turns


def turns_to_fish(turns):
    return "\n".join(f"<|speaker:{sid}|> {text}" for sid, text in turns)


turns = parse_turns(SCRIPT_FILE.read_text(encoding="utf-8"))
assert turns, "No Speaker 1/2 or [M]/[F] lines found"

FISH_TEXT = turns_to_fish(turns)
if TURNS_PER_CHUNK and TURNS_PER_CHUNK > 0:
    FISH_CHUNKS = [
        turns_to_fish(turns[i : i + TURNS_PER_CHUNK])
        for i in range(0, len(turns), TURNS_PER_CHUNK)
    ]
else:
    FISH_CHUNKS = [FISH_TEXT]
FISH_SMOKE_TEXT = turns_to_fish(turns[:SMOKE_TURNS])

total_words = sum(len(t.split()) for _, t in turns)
print(f"{len(turns)} turns, {total_words} words → {len(FISH_CHUNKS)} generate pass(es)")
print(f"Smoke: first {min(SMOKE_TURNS, len(turns))} turns")
print("Preview:")
print(FISH_TEXT[:400], "..." if len(FISH_TEXT) > 400 else "")

In [ ]:
# 8. CLI helper + smoke generate (first SMOKE_TURNS)
# SHARED helper — if the CLI invocation errors, patch BOTH notebooks.
import gc
import os
import site
import subprocess
import sys
from pathlib import Path

import torch
import IPython.display as ipd

assert torch.cuda.is_available(), "GPU lost — nvidia-smi should still work on this VM"
gc.collect()
torch.cuda.empty_cache()
print("VRAM before generate:", round(torch.cuda.memory_allocated() / 1e9, 2), "GB allocated")

OUT_DIR.mkdir(parents=True, exist_ok=True)
INFER = FISH_REPO / "fish_speech" / "models" / "text2semantic" / "inference.py"
assert INFER.exists(), INFER


def run_fish_inference(text: str, out_wav: Path) -> None:
    cmd = [
        sys.executable,
        str(INFER),
        "--text", text,
        "--prompt-audio", str(SPEAKER0_REF),
        "--prompt-text", REF_TEXT_M,
        "--prompt-audio", str(SPEAKER1_REF),
        "--prompt-text", REF_TEXT_F,
        "--checkpoint-path", str(CKPT_DIR),
        "--output", str(out_wav),
        "--output-dir", str(OUT_DIR / "codes"),
        "--device", "cuda",
        "--seed", str(SEED),
        "--chunk-length", str(CHUNK_LENGTH),
    ]
    cmd.append("--compile" if USE_COMPILE else "--no-compile")
    if USE_HALF:
        cmd.append("--half")
    env = os.environ.copy()
    env["PYTHONPATH"] = os.pathsep.join([
        site.getusersitepackages(),
        str(FISH_REPO),
        env.get("PYTHONPATH", ""),
    ])
    print("Running:", " ".join(cmd[:6]), "... →", out_wav.name)
    proc = subprocess.run(
        cmd,
        cwd=str(FISH_REPO),
        env=env,
        capture_output=True,
        text=True,
    )
    if proc.stdout:
        print(proc.stdout[-4000:])
    if proc.returncode != 0:
        print(proc.stderr[-6000:] if proc.stderr else "(no stderr)")
        raise RuntimeError(f"inference.py failed ({proc.returncode}) for {out_wav.name}")
    assert out_wav.exists() and out_wav.stat().st_size > 1000, out_wav
    print(f"  -> {out_wav} ({out_wav.stat().st_size / 1e6:.1f} MB)")


SMOKE_PATH = OUT_DIR / "smoke.wav"
print(f"=== smoke ({SMOKE_TURNS} turns) ===")
run_fish_inference(FISH_SMOKE_TEXT, SMOKE_PATH)
ipd.Audio(str(SMOKE_PATH))

In [ ]:
# 9. Full episode generate (run smoke first so run_fish_inference exists)
import gc

import torch

assert "run_fish_inference" in dir(), "Run the smoke generate cell first (defines the CLI helper)."
gc.collect()
torch.cuda.empty_cache()

EPISODE_GEN = OUT_DIR / "episode.wav"
print(f"=== full episode ({len(turns)} turns) ===")
run_fish_inference(FISH_TEXT, EPISODE_GEN)
print("Full generate finished:", EPISODE_GEN)

In [ ]:
# 10. Assemble episode + listen
from pathlib import Path

import IPython.display as ipd
from pydub import AudioSegment

OUT_DIR = Path.home() / "fish-s2" / "outputs"
chunks = sorted(OUT_DIR.glob("chunk_*.wav"))
direct = OUT_DIR / "episode.wav"
if chunks:
    clips = chunks
elif direct.exists() and direct.stat().st_size > 1000:
    clips = [direct]
else:
    clips = [
        p for p in sorted(OUT_DIR.glob("*.wav"))
        if p.stat().st_size > 1000 and p.name not in {"episode.wav", "smoke.wav"}
    ]
assert clips, f"No wav outputs in {OUT_DIR} — scroll generate cell logs for errors"
print("Found clips:", [c.name for c in clips])

episode = AudioSegment.empty()
for clip in clips:
    episode += AudioSegment.from_wav(clip)

EPISODE_PATH = OUT_DIR / "episode.wav"
episode.export(EPISODE_PATH, format="wav")
print(f"{len(clips)} clip(s) → {EPISODE_PATH} ({len(episode) / 60000:.1f} min)")
ipd.Audio(str(EPISODE_PATH))

In [ ]:
# 11. Pull to Mac (no Colab download) then mix locally
print("episode:", EPISODE_PATH)
print(
    "\nOn your Mac:\n"
    "  gcloud compute scp $VM:~/fish-s2/outputs/episode.wav . --zone=$ZONE\n"
    "  python run_mix_moss.py --episode episode.wav "
    "--cues output/chokhi_dhani_fish.cues.json -o output/episode_mixed.mp3"
)

## Troubleshooting (S2 Pro L4 VM Jupyter)

- **Install / parse / CLI errors** — patch **both** notebooks unless the fix is an L4-only knob.
- **Do not `pip install -e .`** — Ubuntu `vcs-versioning` / `packaging 21.3` conflict. Use the `.pth` + PYTHONPATH path in cell 4.
- **Hydra `config["class"]` / `rpartition`** — `hydra-core` must be **1.3.2**, not 0.11.3. Cell 4 uninstalls 0.11 and pins 1.3.2. Restart the kernel after that install.
- **`No module named 'audiotools'` / `'dac'`** — `descript-audiotools` and `descript-audio-codec`.
- **torchaudio `undefined symbol`** — `/usr/local` torchaudio ≠ VM torch. Cell 4 installs matching torchaudio into **user site**; smoke sets PYTHONPATH user-site first. Never `pip install torch`.
- **`TorchCodec` / `libnvrtc.so.13`** — do not install torchcodec. Cell 4 patches `encode_audio` to **soundfile**.
- **No directed .txt** — SCP `output/chokhi_dhani_fish.txt` into `~/fish-s2/inputs/`.
- **Assert failed (VRAM / capability)** — T4 VM. Use `fish_s2_t4_jupyter.ipynb`.
- **Do not use SGLang** in v1. Official `inference.py` only.
- **`--compile`** — leave False for the first smoke.
- **Holds / ambience** — `run_mix_moss.py` on Mac with `output/chokhi_dhani_fish.cues.json`.